In [2]:
src_lang = "ind"
target_lang = ["aaz", "ptu", "nfa", "heg", "lex", "row", "llg", "rgu", "txq", "tet", "wrs"]
NT_BOOKS = [
    "MAT",
    "MRK",
    "LUK",
    "JHN",
    "ACT",
    "ROM",
    "1CO",
    "2CO",
    "GAL",
    "EPH",
    "PHP",
    "COL",
    "1TH",
    "2TH",
    "1TI",
    "2TI",
    "TIT",
    "PHM",
    "HEB",
    "JAS",
    "1PE",
    "2PE",
    "1JN",
    "2JN",
    "3JN",
    "JUD",
    "REV",
]

In [22]:
from datasets import load_dataset

dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=["ind"], trust_remote_code=True)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 7408
    })
    validation: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 415
    })
    test: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 406
    })
})


In [46]:
print(dataset['train'][0])

{'translation': {'language': ['aaz', 'ind'], 'translation': ['Au ꞌtoit he Usif Yesus nakriraꞌ In nekan arekot neu Iin na ok-okeꞌ. Tebes namneo. Amin. Naꞌko au, Naiꞌ Yohanis kau, tua.', 'Semoga kasih karunia Tuhan Yesus bersama dengan orang-orang percaya. Amin.']}, 'files': {'lang': ['aaz', 'ind'], 'file': ['aaz-aaz.txt', 'ind-indags.txt']}, 'ref': ['REV 22:21'], 'licenses': ['http://creativecommons.org/licenses/by-nd/4.0/', 'http://creativecommons.org/licenses/by-sa/4.0/'], 'copyrights': ['Unit Bahasa dan Budaya, Kupang NTT, Indonesia', '']}


In [5]:
for datum in dataset["train"]:
    if len(datum["ref"]) > 1:
        print(f"Ref: {datum['ref']}")
        print(f"Files: {datum['files']}")
        print(f"Licenses: {datum['licenses']}")
        print(f"Language: {datum['translation']['language']}")
        print(f"Translation: {datum['translation']['translation']}")
        print(f"Translation length: {len(datum['translation']['translation'])}\n")
        
        break

Ref: ['ACT 21:23', 'ACT 21:24']
Files: {'lang': ['aaz', 'ind', 'ind'], 'file': ['aaz-aaz.txt', 'ind-ind.txt', 'ind-indags.txt']}
Licenses: ['http://creativecommons.org/licenses/by-nd/4.0/', 'http://creativecommons.org/licenses/by-nd/4.0/', 'http://creativecommons.org/licenses/by-sa/4.0/']
Language: ['aaz', 'ind', 'ind']
Translation: ['Onaim karu reko te, ho mtaam mok sin meu Uisneno In Uim Onen Uuf. Ho mutuin sin amsaꞌ, he mmoeꞌ mutuinaꞌ harat reꞌ he mukninuꞌ ho tuam aan. Ho ro he msutai sin baen reꞌ natuin harat amsaꞌ. Karu ho mmoeꞌ on reꞌ naan, of too mfaun ein nak, ho mnaaꞌ muher-heran prenat pirsait Yahudi feꞌ, reꞌ kaꞌo Musa nahakeꞌ sin neu kit. Ma sin of nahinin retaꞌ reꞌ nak, ho mtaar atoniꞌ sin he kais natuin harat rais pirsait Yahudi, naan rais poi ok-okeꞌ!', 'Jadi kami penatua menasihatkan Saudara untuk melakukan ini: Di antara saudara seiman, ada empat orang yang sudah menyelesaikan masa perjanjian khusus dan perlu mengikuti upacara penyucian di teras Rumah Allah. Dukunglah m

In [16]:
from datasets import load_dataset, DatasetDict

def process_translations(x, src_lang: str, tgt_lang: str):
    """
    Select a single source/target translation per row and record chosen files.
    - Source is `src_lang`; if multiple versions exist, prefer '{src_lang}-{src_lang}.txt' (e.g., 'ind-ind.txt').
    - Target is `tgt_lang`; assumed unique in the row.
    Returns: text_source, text_target, source_file, target_file
    """
    tr = x.get("translation") or {}
    languages = list(tr.get("language") or [])
    texts = list(tr.get("translation") or [])

    files_info = x.get("files") or {}
    file_names = files_info.get("file") if isinstance(files_info, dict) else None

    # Select source (prefer '{src}-{src}.txt' when multiple)
    src_indices = [i for i, lang in enumerate(languages) if lang == src_lang]
    selected_source_idx = None
    if src_indices:
        if len(src_indices) == 1:
            selected_source_idx = src_indices[0]
        else:
            preferred_idx = None
            if isinstance(file_names, list) and len(file_names) == len(languages):
                preferred_filename = f"{src_lang}-{src_lang}.txt"
                for idx in src_indices:
                    if file_names[idx] == preferred_filename:
                        preferred_idx = idx
                        break
            selected_source_idx = preferred_idx if preferred_idx is not None else src_indices[0]

    # Select target (first occurrence of tgt_lang)
    tgt_indices = [i for i, lang in enumerate(languages) if lang == tgt_lang]
    selected_target_idx = tgt_indices[0] if tgt_indices else None

    source_text = texts[selected_source_idx] if selected_source_idx is not None and selected_source_idx < len(texts) else ""
    target_text = texts[selected_target_idx] if selected_target_idx is not None and selected_target_idx < len(texts) else ""

    source_file = ""
    target_file = ""
    if isinstance(file_names, list) and len(file_names) == len(languages):
        if selected_source_idx is not None and selected_source_idx < len(file_names):
            source_file = file_names[selected_source_idx]
        if selected_target_idx is not None and selected_target_idx < len(file_names):
            target_file = file_names[selected_target_idx]

    return {
        "text_source": source_text,
        "text_target": target_text,
        "source_file": source_file,
        "target_file": target_file,
    }
    

def load_ebible_corpus(src_lang, tgt_lang):
    dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=[src_lang, tgt_lang], trust_remote_code=True)
    dataset = dataset.map(process_translations, fn_kwargs={"src_lang": src_lang, "tgt_lang": tgt_lang})
    # OT books for testing, NT books for training and validation
    # Handle both single refs and multiple refs
    def is_nt_book(refs):
        if isinstance(refs, list):
            # Check if any ref belongs to NT books
            return any(ref.split()[0] in NT_BOOKS for ref in refs)
        else:
            # Single reference
            return refs.split()[0] in NT_BOOKS
    
    # The dataset is a DatasetDict, so we need to access the 'train' split
    train_data = dataset['train']
    train_ds = train_data.filter(lambda x: is_nt_book(x["ref"]))
    test_ds = train_data.filter(lambda x: not is_nt_book(x["ref"]))
    train_val_ds = train_ds.train_test_split(test_size=0.1, seed=41)
    dataset = DatasetDict({"train": train_val_ds["train"], "validation": train_val_ds["test"], "test": test_ds})
    return dataset

dataset = load_ebible_corpus("ind", "aaz")

# # Process all target languages and combine into one dataset
# all_datasets = {}

# for tgt_lang in target_lang:
#     print(f"Processing {source_lang} -> {tgt_lang}...")
#     dataset = load_ebible_corpus(source_lang, tgt_lang)

# # Create subset names for each split
# subset_name = f"{source_lang}-{tgt_lang}"

# # Add each split with the language pair prefix
# for split_name, split_data in dataset.items():
#     full_subset_name = f"{subset_name}-{split_name}"
#     all_datasets[full_subset_name] = split_data
#     print(f"  Added {full_subset_name}: {len(split_data)} examples")

# # Combine all into one DatasetDict
# combined_dataset = DatasetDict(all_datasets)

# print(f"\nFinal combined dataset structure:")
# for subset_name, subset_data in combined_dataset.items():
#     print(f"  {subset_name}: {len(subset_data)} examples")


In [26]:
for datum in dataset["train"]:
    if len(datum["ref"]) > 1:
        print(f"Ref: {datum['ref']}")
        print(f"Files: {datum['files']}")
        print(f"Licenses: {datum['licenses']}")
        print(f"Language: {datum['translation']['language']}")
        print(f"Translation: {datum['translation']['translation']}")
        # print(f"Source file: {datum['source_file']}")
        # print(f"Target file: {datum['target_file']}")
        # print(f"Source: {datum['text_source']}")
        # print(f"Target: {datum['text_target']}\n")
        
        # break

Ref: ['REV 21:12', 'REV 21:13']
Files: {'lang': ['ind', 'ind'], 'file': ['ind-indags.txt', 'ind-ind.txt']}
Licenses: ['http://creativecommons.org/licenses/by-sa/4.0/', 'http://creativecommons.org/licenses/by-nd/4.0/']
Language: ['ind', 'ind']
Translation: ['Ada tiga pintu gerbang di masing-masing sisi tembok itu, yaitu di sebelah timur, utara, selatan dan barat.', 'Kota itu mempunyai tembok yang besar dan tinggi sekali. Dan tembok itu mempunyai dua belas pintu gerbang— tiga pintu gerbang pada setiap sisinya, tiga pintu di sebelah timur, tiga pintu di sebelah utara, tiga pintu di sebelah selatan, dan tiga pintu di sebelah barat. Dan setiap pintu dijaga oleh satu malaikat. Di atas setiap pintu itu tertulis masing-masing satu nama dari nama-nama kedua belas suku Israel.']
Ref: ['REV 20:14', 'REV 20:15']
Files: {'lang': ['ind', 'ind'], 'file': ['ind-indags.txt', 'ind-ind.txt']}
Licenses: ['http://creativecommons.org/licenses/by-sa/4.0/', 'http://creativecommons.org/licenses/by-nd/4.0/']
La

In [14]:
# Push the combined dataset to Hugging Face Hub
def push_to_hub(dataset, repo_name, private=False):
    """
    Push the processed dataset to Hugging Face Hub
    
    Args:
        dataset: The DatasetDict to push
        repo_name: Name of the repository (e.g., "username/dataset-name")
        private: Whether to make the repository private
    """
    try:
        # Push the dataset
        dataset.push_to_hub(
            repo_id=repo_name,
            private=private,
            token=True  # Uses your saved HF token
        )
        print(f"✅ Successfully pushed dataset to: https://huggingface.co/datasets/{repo_name}")
        
        # Print dataset card information
        print(f"\n📝 Dataset structure:")
        for subset_name in dataset.keys():
            lang_pair = subset_name.rsplit('-', 1)[0]  # Remove split suffix
            split = subset_name.rsplit('-', 1)[1]      # Get split name
            print(f"  {subset_name}: {len(dataset[subset_name])} examples")
            
    except Exception as e:
        print(f"❌ Error pushing to hub: {e}")
        print("Make sure you're logged in with `huggingface-cli login`")

# Example usage (uncomment and modify as needed):
# REPO_NAME = "your-username/bible-nmt-multilingual"  # Change this to your desired repo name
# push_to_hub(combined_dataset, REPO_NAME, private=False)

print("Dataset is ready to be pushed to Hub!")
print("Uncomment and modify the REPO_NAME above, then run the push_to_hub function.")


Dataset is ready to be pushed to Hub!
Uncomment and modify the REPO_NAME above, then run the push_to_hub function.


In [15]:
combined_dataset.push_to_hub("biblenlp-corpus")

ValueError: Split name should match '^\w+(\.\w+)*$' but got 'ind-wrs-train'.